### Set Explicit Data Directory and Load pyspark

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("data-modeling-data-vault")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 21:36:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 21:36:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.5.4


## Data Vault 2.0 Model

Business keys:
- Customer: customer_id
- Order: order_id
- Product: product_id

Hubs:
- hub_customer
- hub_order
- hub_product

Links:
- link_customer_order
- link_order_product

Satellites:
- sat_customer
- sat_order
- sat_product
- sat_order_item

### Creating customer hub and satellite

In [3]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

bronze_customers.show()

+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_id|first_name|last_name|               email|       city|state|         created_at|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [4]:
hub_customer = (
    bronze_customers
    .select("customer_id")
    .distinct()
    .withColumn(
        "customer_hk",
        F.md5(F.col("customer_id").cast("string"))
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("crm.customers"))
    .select(
        "customer_hk",
        "customer_id",
        "load_date",
        "record_source"
    )
)

hub_customer.show(truncate=False)

+--------------------------------+-----------+-----------------------+-------------+
|customer_hk                     |customer_id|load_date              |record_source|
+--------------------------------+-----------+-----------------------+-------------+
|fba9d88164f3e2d9109ee770223212a0|1002       |2026-10-04 21:36:46.782|crm.customers|
|b8c37e33defde51cf91e1e03e51657da|1001       |2026-10-04 21:36:46.782|crm.customers|
|aa68c75c4a77c87f97fb686b2f068676|1003       |2026-10-04 21:36:46.782|crm.customers|
|fed33392d3a48aa149a87a38b875ba4a|1004       |2026-10-04 21:36:46.782|crm.customers|
+--------------------------------+-----------+-----------------------+-------------+



In [5]:
sat_customer = (
    bronze_customers
    .withColumn(
        "customer_hk",
        F.md5(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "hashdiff",
        F.md5(
            F.concat_ws(
                "||",
                F.coalesce(F.col("first_name"), F.lit("")),
                F.coalesce(F.col("last_name"), F.lit("")),
                F.coalesce(F.col("email"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("state"), F.lit(""))
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("crm.customers"))
    .select(
        "customer_hk",
        "hashdiff",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "load_date",
        "record_source"
    )
)

sat_customer.show(truncate=False)

+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-------------------+-------------+
|customer_hk                     |hashdiff                        |first_name|last_name|email                   |city       |state|load_date          |record_source|
+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-------------------+-------------+
|b8c37e33defde51cf91e1e03e51657da|c3b33848b0d75d093a628c78e0c58a5a|Alice     |Nguyen   |alice.nguyen@example.com|Los Angeles|CA   |2026-10-04 21:36:48|crm.customers|
|fba9d88164f3e2d9109ee770223212a0|f0767e068d6a565cc2852e2b8a9732bc|Marcus    |Hill     |marcus.hill@example.com |Austin     |TX   |2026-10-04 21:36:48|crm.customers|
|aa68c75c4a77c87f97fb686b2f068676|f66f3839956eb97fea0d1ee44fce7f5f|Priya     |Shah     |priya.shah@example.com  |Seattle    |WA   |2026-10-04 21:36:48|crm.customers|
|fed

In [6]:
hub_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "hub_customer")
)

sat_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
)

In [7]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_customer")
).show(truncate=False)

spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
).show(truncate=False)

+--------------------------------+-----------+-----------------------+-------------+
|customer_hk                     |customer_id|load_date              |record_source|
+--------------------------------+-----------+-----------------------+-------------+
|fba9d88164f3e2d9109ee770223212a0|1002       |2026-10-04 21:36:48.231|crm.customers|
|b8c37e33defde51cf91e1e03e51657da|1001       |2026-10-04 21:36:48.231|crm.customers|
|aa68c75c4a77c87f97fb686b2f068676|1003       |2026-10-04 21:36:48.231|crm.customers|
|fed33392d3a48aa149a87a38b875ba4a|1004       |2026-10-04 21:36:48.231|crm.customers|
+--------------------------------+-----------+-----------------------+-------------+

+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-----------------------+-------------+
|customer_hk                     |hashdiff                        |first_name|last_name|email                   |city       |state|load_date         

### Creating order hub and satellite

In [8]:
bronze_orders = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_orders")
)

bronze_orders.show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



In [9]:
hub_order = (
    bronze_orders
    .select("order_id")
    .distinct()
    .withColumn(
        "order_hk",
        F.md5(F.col("order_id").cast("string"))
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("sales.orders"))
    .select(
        "order_hk",
        "order_id",
        "load_date",
        "record_source"
    )
)

hub_order.show(truncate=False)

+--------------------------------+--------+-----------------------+-------------+
|order_hk                        |order_id|load_date              |record_source|
+--------------------------------+--------+-----------------------+-------------+
|240ac9371ec2671ae99847c3ae2e6384|5003    |2026-10-04 21:36:50.947|sales.orders |
|03b264c595403666634ac75d828439bc|5001    |2026-10-04 21:36:50.947|sales.orders |
|3202111cf90e7c816a472aaceb72b0df|5004    |2026-10-04 21:36:50.947|sales.orders |
|1d6408264d31d453d556c60fe7d0459e|5005    |2026-10-04 21:36:50.947|sales.orders |
|415585bd389b69659223807d77a96791|5002    |2026-10-04 21:36:50.947|sales.orders |
+--------------------------------+--------+-----------------------+-------------+



In [10]:
sat_order = (
    bronze_orders
    .withColumn(
        "order_hk",
        F.md5(F.col("order_id").cast("string"))
    )
    .withColumn(
        "hashdiff",
        F.md5(
            F.concat_ws(
                "||",
                F.coalesce(F.col("order_date").cast("string"), F.lit("")),
                F.coalesce(F.col("order_status"), F.lit("")),
                F.coalesce(F.col("total_amount").cast("string"), F.lit(""))
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("sales.orders"))
    .select(
        "order_hk",
        "hashdiff",
        "order_date",
        "order_status",
        "total_amount",
        "load_date",
        "record_source"
    )
)

sat_order.show(truncate=False)

+--------------------------------+--------------------------------+----------+------------+------------+-----------------------+-------------+
|order_hk                        |hashdiff                        |order_date|order_status|total_amount|load_date              |record_source|
+--------------------------------+--------------------------------+----------+------------+------------+-----------------------+-------------+
|03b264c595403666634ac75d828439bc|7d27477ac9dc4c8ba89aad7a61465f44|2026-03-10|Completed   |129.99      |2026-10-04 21:36:51.243|sales.orders |
|415585bd389b69659223807d77a96791|91e0c27b5d712530b2ae8e92ac29eb7a|2026-03-11|Completed   |84.50       |2026-10-04 21:36:51.243|sales.orders |
|240ac9371ec2671ae99847c3ae2e6384|3c303453482c0041e44d08e493509396|2026-03-14|Pending     |42.00       |2026-10-04 21:36:51.243|sales.orders |
|3202111cf90e7c816a472aaceb72b0df|c136f8a4a5573ac14e2ad6ad10539fae|2026-03-18|Completed   |215.75      |2026-10-04 21:36:51.243|sales.orders |

In [11]:
hub_order.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "hub_order")
)

sat_order.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "sat_order")
)

In [12]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_order")
).show(truncate=False)

spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_order")
).show(truncate=False)

+--------------------------------+--------+-----------------------+-------------+
|order_hk                        |order_id|load_date              |record_source|
+--------------------------------+--------+-----------------------+-------------+
|240ac9371ec2671ae99847c3ae2e6384|5003    |2026-10-04 21:36:51.457|sales.orders |
|03b264c595403666634ac75d828439bc|5001    |2026-10-04 21:36:51.457|sales.orders |
|3202111cf90e7c816a472aaceb72b0df|5004    |2026-10-04 21:36:51.457|sales.orders |
|1d6408264d31d453d556c60fe7d0459e|5005    |2026-10-04 21:36:51.457|sales.orders |
|415585bd389b69659223807d77a96791|5002    |2026-10-04 21:36:51.457|sales.orders |
+--------------------------------+--------+-----------------------+-------------+

+--------------------------------+--------------------------------+----------+------------+------------+-----------------------+-------------+
|order_hk                        |hashdiff                        |order_date|order_status|total_amount|load_date     

### Build customer order link table

In [13]:
link_customer_order = (
    bronze_orders
    .withColumn(
        "customer_hk",
        F.md5(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "order_hk",
        F.md5(F.col("order_id").cast("string"))
    )
    .withColumn(
        "customer_order_hk",
        F.md5(
            F.concat_ws(
                "||",
                F.col("customer_id").cast("string"),
                F.col("order_id").cast("string")
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("sales.orders"))
    .select(
        "customer_order_hk",
        "customer_hk",
        "order_hk",
        "load_date",
        "record_source"
    )
)

link_customer_order.show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-----------------------+-------------+
|customer_order_hk               |customer_hk                     |order_hk                        |load_date              |record_source|
+--------------------------------+--------------------------------+--------------------------------+-----------------------+-------------+
|523e3f5187fd2da616aa8606d36a593d|b8c37e33defde51cf91e1e03e51657da|03b264c595403666634ac75d828439bc|2026-10-04 21:36:52.827|sales.orders |
|1c810ca82a1b08288d21abdec47e4b01|fba9d88164f3e2d9109ee770223212a0|415585bd389b69659223807d77a96791|2026-10-04 21:36:52.827|sales.orders |
|060046d3fe7ec63542e3e3fc741d7e3d|b8c37e33defde51cf91e1e03e51657da|240ac9371ec2671ae99847c3ae2e6384|2026-10-04 21:36:52.827|sales.orders |
|04de6019b8d1fad08f7a020f5063c6e5|fed33392d3a48aa149a87a38b875ba4a|3202111cf90e7c816a472aaceb72b0df|2026-10-04 21:36:52.827|sales.orders |
|9360e153ffe6e66a033dfc9193

In [14]:
link_customer_order.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "link_customer_order")
)

In [15]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "link_customer_order")
).show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-----------------------+-------------+
|customer_order_hk               |customer_hk                     |order_hk                        |load_date              |record_source|
+--------------------------------+--------------------------------+--------------------------------+-----------------------+-------------+
|523e3f5187fd2da616aa8606d36a593d|b8c37e33defde51cf91e1e03e51657da|03b264c595403666634ac75d828439bc|2026-10-04 21:36:53.031|sales.orders |
|1c810ca82a1b08288d21abdec47e4b01|fba9d88164f3e2d9109ee770223212a0|415585bd389b69659223807d77a96791|2026-10-04 21:36:53.031|sales.orders |
|060046d3fe7ec63542e3e3fc741d7e3d|b8c37e33defde51cf91e1e03e51657da|240ac9371ec2671ae99847c3ae2e6384|2026-10-04 21:36:53.031|sales.orders |
|04de6019b8d1fad08f7a020f5063c6e5|fed33392d3a48aa149a87a38b875ba4a|3202111cf90e7c816a472aaceb72b0df|2026-10-04 21:36:53.031|sales.orders |
|9360e153ffe6e66a033dfc9193

hub_customer ──┐
               ├── link_customer_order
hub_order ─────┘

hub_customer → sat_customer
hub_order    → sat_order

### Create product hub and satellite

In [16]:
bronze_products = spark.read.parquet(
    str(DATA_DIR / "bronze" / "erp_products")
)

bronze_products.show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



In [17]:
hub_product = (
    bronze_products
    .select("product_id")
    .distinct()
    .withColumn(
        "product_hk",
        F.md5(F.col("product_id").cast("string"))
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("erp.products"))
    .select(
        "product_hk",
        "product_id",
        "load_date",
        "record_source"
    )
)

hub_product.show(truncate=False)

+--------------------------------+----------+-----------------------+-------------+
|product_hk                      |product_id|load_date              |record_source|
+--------------------------------+----------+-----------------------+-------------+
|a591024321c5e2bdbd23ed35f0574dde|2003      |2026-10-04 21:36:54.084|erp.products |
|b8b4b727d6f5d1b61fff7be687f7970f|2004      |2026-10-04 21:36:54.084|erp.products |
|d0fb963ff976f9c37fc81fe03c21ea7b|2001      |2026-10-04 21:36:54.084|erp.products |
|4ba29b9f9e5732ed33761840f4ba6c53|2002      |2026-10-04 21:36:54.084|erp.products |
+--------------------------------+----------+-----------------------+-------------+



In [18]:
sat_product = (
    bronze_products
    .withColumn(
        "product_hk",
        F.md5(F.col("product_id").cast("string"))
    )
    .withColumn(
        "hashdiff",
        F.md5(
            F.concat_ws(
                "||",
                F.coalesce(F.col("product_name"), F.lit("")),
                F.coalesce(F.col("category"), F.lit("")),
                F.coalesce(F.col("unit_cost").cast("string"), F.lit("")),
                F.coalesce(F.col("unit_price").cast("string"), F.lit(""))
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("erp.products"))
    .select(
        "product_hk",
        "hashdiff",
        "product_name",
        "category",
        "unit_cost",
        "unit_price",
        "load_date",
        "record_source"
    )
)

sat_product.show(truncate=False)

+--------------------------------+--------------------------------+-------------------+-----------+---------+----------+-----------------------+-------------+
|product_hk                      |hashdiff                        |product_name       |category   |unit_cost|unit_price|load_date              |record_source|
+--------------------------------+--------------------------------+-------------------+-----------+---------+----------+-----------------------+-------------+
|d0fb963ff976f9c37fc81fe03c21ea7b|8e4102d29268d98b098cf31efb8f3bcb|Wireless Mouse     |Electronics|18.00    |34.99     |2026-10-04 21:36:54.449|erp.products |
|4ba29b9f9e5732ed33761840f4ba6c53|9845ef6d7c11b287afcced38780aa0da|Mechanical Keyboard|Electronics|52.00    |89.99     |2026-10-04 21:36:54.449|erp.products |
|a591024321c5e2bdbd23ed35f0574dde|4f0e75ca22ed0fd760618ee8abeb4cea|Laptop Stand       |Office     |21.50    |44.99     |2026-10-04 21:36:54.449|erp.products |
|b8b4b727d6f5d1b61fff7be687f7970f|35a37dbfff39

In [19]:
hub_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "hub_product")
)

sat_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "sat_product")
)

In [20]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "hub_product")
).show(truncate=False)

spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_product")
).show(truncate=False)

+--------------------------------+----------+-----------------------+-------------+
|product_hk                      |product_id|load_date              |record_source|
+--------------------------------+----------+-----------------------+-------------+
|a591024321c5e2bdbd23ed35f0574dde|2003      |2026-10-04 21:36:54.637|erp.products |
|b8b4b727d6f5d1b61fff7be687f7970f|2004      |2026-10-04 21:36:54.637|erp.products |
|d0fb963ff976f9c37fc81fe03c21ea7b|2001      |2026-10-04 21:36:54.637|erp.products |
|4ba29b9f9e5732ed33761840f4ba6c53|2002      |2026-10-04 21:36:54.637|erp.products |
+--------------------------------+----------+-----------------------+-------------+

+--------------------------------+--------------------------------+-------------------+-----------+---------+----------+-----------------------+-------------+
|product_hk                      |hashdiff                        |product_name       |category   |unit_cost|unit_price|load_date              |record_source|
+--------

### Build order product link table and satellite

In [21]:
bronze_order_items = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_order_items")
)

link_order_product = (
    bronze_order_items
    .withColumn(
        "order_hk",
        F.md5(F.col("order_id").cast("string"))
    )
    .withColumn(
        "product_hk",
        F.md5(F.col("product_id").cast("string"))
    )
    .withColumn(
        "order_product_hk",
        F.md5(
            F.concat_ws(
                "||",
                F.col("order_id").cast("string"),
                F.col("product_id").cast("string")
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("sales.order_items"))
    .select(
        "order_product_hk",
        "order_hk",
        "product_hk",
        "load_date",
        "record_source"
    )
)

link_order_product.show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-----------------------+-----------------+
|order_product_hk                |order_hk                        |product_hk                      |load_date              |record_source    |
+--------------------------------+--------------------------------+--------------------------------+-----------------------+-----------------+
|d34e571c2c35dafa02f40a87f127d5c7|03b264c595403666634ac75d828439bc|d0fb963ff976f9c37fc81fe03c21ea7b|2026-10-04 21:36:56.041|sales.order_items|
|99d86acace1c7700efaad3c69dba02d8|03b264c595403666634ac75d828439bc|4ba29b9f9e5732ed33761840f4ba6c53|2026-10-04 21:36:56.041|sales.order_items|
|7df5b18ac6f3ee21cefe91d8bc0fd9d7|415585bd389b69659223807d77a96791|a591024321c5e2bdbd23ed35f0574dde|2026-10-04 21:36:56.041|sales.order_items|
|ab1d36b0091be550ad8e46849a91d153|240ac9371ec2671ae99847c3ae2e6384|b8b4b727d6f5d1b61fff7be687f7970f|2026-10-04 21:36:56.041|sales.order_items|

In [22]:
sat_order_product = (
    bronze_order_items
    .withColumn(
        "order_product_hk",
        F.md5(
            F.concat_ws(
                "||",
                F.col("order_id").cast("string"),
                F.col("product_id").cast("string")
            )
        )
    )
    .withColumn(
        "hashdiff",
        F.md5(
            F.concat_ws(
                "||",
                F.coalesce(F.col("quantity").cast("string"), F.lit("")),
                F.coalesce(F.col("unit_price").cast("string"), F.lit(""))
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("sales.order_items"))
    .select(
        "order_product_hk",
        "hashdiff",
        "quantity",
        "unit_price",
        "load_date",
        "record_source"
    )
)

sat_order_product.show(truncate=False)

+--------------------------------+--------------------------------+--------+----------+-----------------------+-----------------+
|order_product_hk                |hashdiff                        |quantity|unit_price|load_date              |record_source    |
+--------------------------------+--------------------------------+--------+----------+-----------------------+-----------------+
|d34e571c2c35dafa02f40a87f127d5c7|d01f5bfeb2a3991450bfe4582c31135a|1       |34.99     |2026-10-04 21:36:56.197|sales.order_items|
|99d86acace1c7700efaad3c69dba02d8|584c0da5466624a3a8e75895a2713713|1       |89.99     |2026-10-04 21:36:56.197|sales.order_items|
|7df5b18ac6f3ee21cefe91d8bc0fd9d7|cb26ea73747a06d5eade0a436fc2e33b|2       |44.99     |2026-10-04 21:36:56.197|sales.order_items|
|ab1d36b0091be550ad8e46849a91d153|6d50dcae1e83b718c33633bdc59211fc|1       |54.99     |2026-10-04 21:36:56.197|sales.order_items|
|4c3821b5c8320ca873a25c42d34f833d|58a2136fb721ef0c36f999d0db14458a|2       |89.99     |202

In [23]:
link_order_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "link_order_product")
)

sat_order_product.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "sat_order_product")
)

In [24]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "link_order_product")
).show(truncate=False)

spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_order_product")
).show(truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-----------------------+-----------------+
|order_product_hk                |order_hk                        |product_hk                      |load_date              |record_source    |
+--------------------------------+--------------------------------+--------------------------------+-----------------------+-----------------+
|d34e571c2c35dafa02f40a87f127d5c7|03b264c595403666634ac75d828439bc|d0fb963ff976f9c37fc81fe03c21ea7b|2026-10-04 21:36:56.306|sales.order_items|
|99d86acace1c7700efaad3c69dba02d8|03b264c595403666634ac75d828439bc|4ba29b9f9e5732ed33761840f4ba6c53|2026-10-04 21:36:56.306|sales.order_items|
|7df5b18ac6f3ee21cefe91d8bc0fd9d7|415585bd389b69659223807d77a96791|a591024321c5e2bdbd23ed35f0574dde|2026-10-04 21:36:56.306|sales.order_items|
|ab1d36b0091be550ad8e46849a91d153|240ac9371ec2671ae99847c3ae2e6384|b8b4b727d6f5d1b61fff7be687f7970f|2026-10-04 21:36:56.306|sales.order_items|